# Spice to GDS — AI Agentic Analog Layout
SSCS Chipathon 2026 — gLayout Track (D)

Converts SPICE netlist to DRC-clean GDSII layout using glayout.
Supports AC simulation, DRC/LVS/PEX verification.

PDK: gf180mcuD  |  Supply: 1.8 V
Author: M Taufiqul Huda

In [ ]:
import sys
from pathlib import Path

root = Path("/foss/designs/chipathon2026-D")
sys.path.insert(0, str(root))

from glayout import gf180, sky130
from gdsfactory import Component

from core import (
    clean_param, display_gds, display_component,
    set_pdk, spice_to_gds, llm_to_gds, generate_netlist_from_prompt,
    add_power_strips, add_double_guardring,
    run_drc, run_lvs, run_pex,
    run_ota_ac, run_comparator_tran, compare_pre_post,
    compare_comp_pre_post,
)

import os
import IPython.display
import gdstk

TEMP_DIR = os.getcwd()
GDS_PATH = os.path.join(TEMP_DIR, "out.gds")
SVG_PATH = os.path.join(TEMP_DIR, "out.svg")

In [ ]:
# ==========================================
# INPUT: SPICE Netlist — 5T OTA (gf180mcuD, 1.8V)
# ==========================================

netlist_input = """
.lib "/foss/pdks/gf180mcuD/libs.tech/ngspice/sm141064.ngspice" typical
.subckt ota_simple vin_p vin_n vout vbias vdd vss
M1 n1 vin_p ntail vss nfet_03v3 W=10u L=1u
M2 vout vin_n ntail vss nfet_03v3 W=10u L=1u
M3 n1 n1 vdd vdd pfet_03v3 W=20u L=1u
M4 vout n1 vdd vdd pfet_03v3 W=20u L=1u
M5 ntail vbias vss vss nfet_03v3 W=15u L=1u
.ends
"""

In [ ]:
# ==========================================
# 1. SPICE → GDS (Layout Generation)
# ==========================================

result = spice_to_gds(netlist_input, mode="analog", add_labels=True)
result.write_gds(GDS_PATH)
print(f"[DONE] Layout written to {GDS_PATH}")
display_component(result, scale=2)

In [ ]:
# ==========================================
# 2. Pre-Simulation (AC Analysis)
# ==========================================

pre = run_ota_ac(netlist_input, "ota_simple",
                  vdd=1.8, vcm=0.9, vbias=0.65, cload=2e-12)

print(f"[PRE]  DC Gain={pre['dc_gain_db']:.1f} dB  "
      f"GBW={pre['gbw_hz']/1e6:.1f} MHz  "
      f"PM={pre['phase_margin_deg']:.1f} deg")

# Sweep vbias for optimal bias point:
# for vb in [0.50, 0.55, 0.60, 0.65, 0.70]:
#     r = run_ota_ac(netlist_input, "ota_simple",
#                     vdd=1.8, vcm=0.9, vbias=vb)
#     print(f"vbias={vb:.2f}  DC={r['dc_gain_db']:.1f}dB  GBW={r['gbw_hz']/1e6:.1f}MHz")

In [ ]:
# ==========================================
# 3. DRC / LVS / PEX (Post-Layout)
# ==========================================
# Requires: PDK_ROOT=/foss/pdks  PDK=gf180mcuD  PDKPATH=/foss/pdks/gf180mcuD

import os
os.environ.setdefault("PDK_ROOT", "/foss/pdks")
os.environ.setdefault("PDK", "gf180mcuD")
os.environ.setdefault("PDKPATH", os.path.join(os.environ["PDK_ROOT"], os.environ["PDK"]))
os.environ.setdefault("STD_CELL_LIBRARY", "gf180mcu_fd_sc_mcu7t5v0")

# Run with post-layout verification:
# result = spice_to_gds(netlist_input, mode="analog", add_labels=True,
#                       run_checks=True)

# Or individually:
# drc = run_drc("ota_simple.gds", cell_name="ota_simple")
# print(f"DRC clean: {drc['clean']}")

# lvs = run_lvs("ota_simple.gds", netlist_content=netlist_input,
#                cell_name="ota_simple")
# print(f"LVS match: {lvs['match']}")

# pex = run_pex("ota_simple.gds", cell_name="ota_simple", mode=2)
# print(f"PEX: {pex['pex_path']}")

# Compare pre vs post (C-coupled PEX):
# cmp = compare_pre_post(netlist_input, pex["pex_path"], "ota_simple",
#                         vdd=1.8, vcm=0.9, vbias=0.65)
# print(f"PRE:  {cmp['pre']['dc_gain_db']:.1f} dB")
# print(f"POST: {cmp['post']['dc_gain_db']:.1f} dB")
# print(f"Delta: {cmp['delta'].get('dc_gain_db', 'N/A')}")

In [ ]:
# ==========================================
# 4. LLM → Netlist → GDS (AI Agentic)
# ==========================================

# For AI coding agents (opencode, claude-code, etc.):
# netlist = generate_netlist_from_prompt(
#     "Design a 5T OTA with differential input, PMOS current mirror load, "
#     "and NMOS tail current. Use gf180mcuD PDK, 1.8V supply. "
#     "Target: DC gain > 40 dB, GBW > 1 MHz, PM > 60 deg."
# )
#
# if netlist:
#     result = spice_to_gds(netlist, mode="analog", add_labels=True)
#     result.write_gds("out.gds")